# TP 1 — LDA/QDA y optimización matemática de modelos

## Integrantes
- Jonatan Mild
- Valentin Torres
- Victor Astorga
- Franco Morero
- Francisco Meaca

**Materia:** Análisis Matemático para Inteligencia Artificial | **Año:** 2026



## Versiones utilizadas

In [15]:
import sys
import numpy
import scipy

print(f"Python {sys.version}")
print(f"NumPy  {numpy.__version__}")
print(f"SciPy  {scipy.__version__}")

Python 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
NumPy  2.2.6
SciPy  1.15.3


## Código base


In [16]:
import numpy as np
import pandas as pd
import numpy.linalg as LA
from scipy.linalg import cholesky, solve_triangular
from scipy.linalg.lapack import dtrtri


In [17]:
class BaseBayesianClassifier:
  def __init__(self):
    pass

  def _estimate_a_priori(self, y):
    a_priori = np.bincount(y.flatten().astype(int)) / y.size
    return np.log(a_priori)

  def _fit_params(self, X, y):
    # estimate all needed parameters for given model
    raise NotImplementedError()

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    raise NotImplementedError()

  def fit(self, X, y, a_priori=None):
    # if it's needed, estimate a priori probabilities
    self.log_a_priori = self._estimate_a_priori(y) if a_priori is None else np.log(a_priori)

    # now that everything else is in place, estimate all needed parameters for given model
    self._fit_params(X, y)

  def predict(self, X):
    # this is actually an individual prediction encased in a for-loop
    m_obs = X.shape[1]
    y_hat = np.empty(m_obs, dtype=int)

    for i in range(m_obs):
      y_hat[i] = self._predict_one(X[:,i].reshape(-1,1))

    # return prediction as a row vector (matching y)
    return y_hat.reshape(1,-1)

  def _predict_one(self, x):
    # calculate all log posteriori probabilities (actually, +C)
    log_posteriori = [ log_a_priori_i + self._predict_log_conditional(x, idx) for idx, log_a_priori_i
                  in enumerate(self.log_a_priori) ]

    # return the class that has maximum a posteriori probability
    return np.argmax(log_posteriori)

In [18]:
class QDA(BaseBayesianClassifier):

  def _fit_params(self, X, y):
    # estimate each covariance matrix
    self.inv_covs = [LA.inv(np.cov(X[:,y.flatten()==idx], bias=True))
                      for idx in range(len(self.log_a_priori))]
    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    # predict the log(P(x|G=class_idx)), the log of the conditional probability of x given the class
    # this should depend on the model used
    inv_cov = self.inv_covs[class_idx]
    unbiased_x =  x - self.means[class_idx]
    return 0.5*np.log(LA.det(inv_cov)) -0.5 * unbiased_x.T @ inv_cov @ unbiased_x

In [19]:
class TensorizedQDA(QDA):

    def _fit_params(self, X, y):
        # ask plain QDA to fit params
        super()._fit_params(X,y)

        # stack onto new dimension
        self.tensor_inv_cov = np.stack(self.inv_covs)
        self.tensor_means = np.stack(self.means)

    def _predict_log_conditionals(self,x):
        unbiased_x = x - self.tensor_means
        inner_prod = unbiased_x.transpose(0,2,1) @ self.tensor_inv_cov @ unbiased_x

        return 0.5*np.log(LA.det(self.tensor_inv_cov)) - 0.5 * inner_prod.flatten()

    def _predict_one(self, x):
        # return the class that has maximum a posteriori probability
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

In [20]:
class QDA_Chol1(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        LA.inv(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True))
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

In [21]:
class QDA_Chol2(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.Ls = [
        cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True)
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L = self.Ls[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = solve_triangular(L, unbiased_x, lower=True)

    return -np.log(L.diagonal().prod()) -0.5 * (y**2).sum()

In [22]:
class QDA_Chol3(BaseBayesianClassifier):
  def _fit_params(self, X, y):
    self.L_invs = [
        dtrtri(cholesky(np.cov(X[:,y.flatten()==idx], bias=True), lower=True), lower=1)[0]
        for idx in range(len(self.log_a_priori))
    ]

    self.means = [X[:,y.flatten()==idx].mean(axis=1, keepdims=True)
                  for idx in range(len(self.log_a_priori))]

  def _predict_log_conditional(self, x, class_idx):
    L_inv = self.L_invs[class_idx]
    unbiased_x =  x - self.means[class_idx]

    y = L_inv @ unbiased_x

    return np.log(L_inv.diagonal().prod()) -0.5 * (y**2).sum()

## Datasets

In [23]:
from sklearn.datasets import load_iris, fetch_openml, load_wine
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

def get_iris_dataset():
  data = load_iris()
  X_full = data.data
  y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
  return X_full, y_full

def get_penguins_dataset():
    # get data
    df, tgt = fetch_openml(name="penguins", return_X_y=True, as_frame=True, parser='auto')

    # drop non-numeric columns
    df.drop(columns=["island","sex"], inplace=True)

    # drop rows with missing values
    mask = df.isna().sum(axis=1) == 0
    df = df[mask]
    tgt = tgt[mask]

    return df.values, tgt.to_numpy().reshape(-1,1)

def get_wine_dataset():
    # get data
    data = load_wine()
    X_full = data.data
    y_full = np.array([data.target_names[y] for y in data.target.reshape(-1,1)])
    return X_full, y_full

def get_letters_dataset():
    # get data
    letter = fetch_openml('letter', version=1, as_frame=False)
    return letter.data, letter.target.reshape(-1,1)

def label_encode(y_full):
    return LabelEncoder().fit_transform(y_full.flatten()).reshape(y_full.shape)

def split_transpose(X, y, test_size, random_state):
    # X_train, X_test, y_train, y_test but all transposed
    return [elem.T for elem in train_test_split(X, y, test_size=test_size, random_state=random_state)]

## Benchmarking

In [24]:
import time
from tqdm.notebook import tqdm
from numpy.random import RandomState
import tracemalloc

RNG_SEED = 6553

class Benchmark:
    def __init__(self, X, y, n_runs=1000, warmup=100, mem_runs=100, test_sz=0.3, rng_seed=RNG_SEED, same_splits=True):
        self.X = X
        self.y = y
        self.n = n_runs
        self.warmup = warmup
        self.mem_runs = mem_runs
        self.test_sz = test_sz
        self.det = same_splits
        if self.det:
            self.rng_seed = rng_seed
        else:
            self.rng = RandomState(rng_seed)

        self.data = dict()

        print("Benching params:")
        print("Total runs:",self.warmup+self.mem_runs+self.n)
        print("Warmup runs:",self.warmup)
        print("Peak Memory usage runs:", self.mem_runs)
        print("Running time runs:", self.n)
        approx_test_sz = int(self.y.size * self.test_sz)
        print("Train size rows (approx):",self.y.size - approx_test_sz)
        print("Test size rows (approx):",approx_test_sz)
        print("Test size fraction:",self.test_sz)

    def bench(self, model_class, **kwargs):
        name = model_class.__name__
        time_data = np.empty((self.n, 3), dtype=float)  # train_time, test_time, accuracy
        mem_data = np.empty((self.mem_runs, 2), dtype=float)  # train_peak_mem, test_peak_mem
        rng = RandomState(self.rng_seed) if self.det else self.rng


        for i in range(self.warmup):
            # Instantiate model with error check for unsupported parameters
            model = model_class(**kwargs)

            # Generate current train-test split
            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )
            # Run training and prediction (timing or memory measurement not recorded)
            model.fit(X_train, y_train)
            model.predict(X_test)

        for i in tqdm(range(self.mem_runs), total=self.mem_runs, desc=f"{name} (MEM)"):

            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            tracemalloc.start()

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()

            _, train_peak = tracemalloc.get_traced_memory()
            tracemalloc.reset_peak()

            model.predict(X_test)
            t3 = time.perf_counter()
            _, test_peak = tracemalloc.get_traced_memory()
            tracemalloc.stop()

            mem_data[i,] = (
                train_peak / (1024 * 1024),
                test_peak / (1024 * 1024)
            )

        for i in tqdm(range(self.n), total=self.n, desc=f"{name} (TIME)"):
            model = model_class(**kwargs)

            X_train, X_test, y_train, y_test = split_transpose(
                self.X, self.y,
                test_size=self.test_sz,
                random_state=rng
            )

            t1 = time.perf_counter()
            model.fit(X_train, y_train)
            t2 = time.perf_counter()
            preds = model.predict(X_test)
            t3 = time.perf_counter()

            time_data[i,] = (
                (t2 - t1) * 1000,
                (t3 - t2) * 1000,
                (y_test.flatten() == preds.flatten()).mean()
            )

        self.data[name] = (time_data, mem_data)

    def summary(self, baseline=None):
        aux = []
        for name, (time_data, mem_data) in self.data.items():
            result = {
                'model': name,
                'train_median_ms': np.median(time_data[:, 0]),
                'train_std_ms': time_data[:, 0].std(),
                'test_median_ms': np.median(time_data[:, 1]),
                'test_std_ms': time_data[:, 1].std(),
                'mean_accuracy': time_data[:, 2].mean(),
                'train_mem_median_mb': np.median(mem_data[:, 0]),
                'train_mem_std_mb': mem_data[:, 0].std(),
                'test_mem_median_mb': np.median(mem_data[:, 1]),
                'test_mem_std_mb': mem_data[:, 1].std()
            }
            aux.append(result)
        df = pd.DataFrame(aux).set_index('model')

        if baseline is not None and baseline in self.data:
            df['train_speedup'] = df.loc[baseline, 'train_median_ms'] / df['train_median_ms']
            df['test_speedup'] = df.loc[baseline, 'test_median_ms'] / df['test_median_ms']
            df['train_mem_reduction'] = df.loc[baseline, 'train_mem_median_mb'] / df['train_mem_median_mb']
            df['test_mem_reduction'] = df.loc[baseline, 'test_mem_median_mb'] / df['test_mem_median_mb']
        return df

# Consigna QDA

**Notación**: en general notamos

* $k$ la cantidad de clases
* $n$ la cantidad de observaciones
* $p$ la cantidad de features/variables/predictores


## Tensorización


### 1) Diferencias entre `QDA`y `TensorizedQDA`

1. ¿Sobre qué paraleliza `TensorizedQDA`? ¿Sobre las $k$ clases, las $n$ observaciones a predecir, o ambas?


Paraleliza sobre las $k$ clases (evalúa las k log-verosimilitudes condicionales en un paso tensorial), porque se puede ver que el `predict` original de la clase base `BaseBayesianClassifier` aun mantiene el for loop sobre las $n$ observaciones. Solo se eliminó el for-loop interno sobre las $k$ clases.

2. Analizar los shapes de `tensor_inv_covs` y `tensor_means` y explicar paso a paso cómo es que `TensorizedQDA` llega a predecir lo mismo que `QDA`.

Sean $k$ clases, $p$ features:

- `self.inv_covs` es una lista de $k$ matrices, cada una de shape `(p, p)`.
- `np.stack(self.inv_covs)` las apila en un nuevo eje 0 → `tensor_inv_cov` shape `(k, p, p)`.
- `self.means` es una lista de $k$ vectores columna, cada uno de shape `(p, 1)`.
- `np.stack(self.means)` → `tensor_means` shape `(k, p, 1)`.

**Paso a paso en la predicción** para una observación $x$ de shape `(p, 1)`:

1. `unbiased_x = x - self.tensor_means` → `(p, 1)` se resta a `(k, p, 1)` por broadcasting → shape `(k, p, 1)`. Cada slice $\delta_j = x - \hat{\mu}_j$.

2. `unbiased_x.transpose(0, 2, 1)` → shape `(k, 1, p)`. Esto es $\delta_j^T$ para cada $j$.

3. El producto matricial por bloques:
$$\underbrace{(k,1,p)}_{\delta^T} \cdot \underbrace{(k,p,p)}_{\hat{\Sigma}^{-1}} \cdot \underbrace{(k,p,1)}_{\delta} = \underbrace{(k,1,1)}_{\text{forma cuadrática por clase}}$$

   El `.flatten()` final produce el vector de formas cuadráticas de shape `(k,)`.

4. `0.5 * np.log(LA.det(self.tensor_inv_cov))` aplica `det` a cada matriz en el eje 0 → shape `(k,)`.

5. La resta de ambos vectores da el vector de log-verosimilitudes condicionales.

6. En `_predict_one`, `self.log_a_priori + self._predict_log_conditionals(x)` suma log-priors y log-likelihoods → `np.argmax` devuelve la clase predicha.

El resultado es idéntico al de `QDA` pero sin el `for` sobre clases.


### Optimización

3. Implementar el modelo `FasterQDA` de manera de eliminar el ciclo for en el método predict.

In [25]:
# FasterQDA elimina el for-loop sobre observaciones de predict.
# El flujo es:
#   1. Centrar: U = X - μ_j → (k, p, n) por broadcasting
#   2. Producto: Σ_j⁻¹ U → (k, p, n)
#   3. Forma cuadrática: Uᵀ Σ⁻¹ U → (k, n, n) — acá aparece la matriz n×n
#   4. Diagonal: se extrae con np.diagonal → (k, n) — solo las distancias²
#   5. Log-posterior: suma log-prior + log-determinante - ½ distancias² → (k, n)
#   6. Argmax: por columna (axis=0) → clase ganadora para cada observación

class FasterQDA(TensorizedQDA):
    def _predict_log_conditionals(self, X):
        # X shape: (p, n), tensor_means shape: (k, p, 1), tensor_inv_cov shape: (k, p, p)
        # 1. Centrar X respecto a cada clase: (p, n) - (k, p, 1) → (k, p, n)

        U = X - self.tensor_means

         # 2. Producto Σ_j⁻¹ @ U para cada clase: (k, p, p) @ (k, p, n) → (k, p, n)

        SU = self.tensor_inv_cov @ U

        # 3. Forma cuadrática Uᵀ Σ⁻¹ U por clase: (k, n, p) @ (k, p, n) → (k, n, n)

        quad = U.transpose(0, 2, 1) @ SU

        # 4. Extraer diagonal (dist² de cada obs): (k, n, n) → (k, n)

        dist = np.diagonal(quad, axis1=1, axis2=2)

         # 5. Log-posterior por clase: (k, 1) - (k, n) → (k, n)
        # LA.det(...) devuelve (k,), lo pasamos a (k,1) para broadcasting correcto.
        log_det = 0.5 * np.log(LA.det(self.tensor_inv_cov)).reshape(-1, 1)

        return log_det - 0.5 * dist


    def predict(self, X):
        
        
       
        log_post = (self.log_a_priori.reshape(-1, 1) + self._predict_log_conditionals(X))

        # 6. Clase con máximo a posteriori para cada obs: (n,) → (1, n)
        return np.argmax(log_post, axis=0).reshape(1, -1)

4. Mostrar dónde aparece la mencionada matriz de $n \times n$, donde $n$ es la cantidad de observaciones a predecir.

Aparece en la forma cuadrática al intentar predecir todas las $n$ observaciones de golpe.
Veamos que:

$$
(x - \mu_j)^T \Sigma_j^{-1} (x - \mu_j) \rightarrow (1, p) \times (p, p) \times (p, 1) = \text{escalar}
$$

Si en vez de una sola $x$ usamos todas las observaciones $X \in \mathbb{R}^{p \times n}$:

$$
U = X - \mu_j \rightarrow (p, n)
$$
$$
U^T \Sigma_j^{-1} U \rightarrow (n, p) \times (p, p) \times (p, n) = (n, n)
$$

Ahí está la matriz $n \times n$.

5. Demostrar que
$$
diag(A \cdot B) = \sum_{cols} A \odot B^T = np.sum(A \odot B^T, axis=1)
$$ es decir, que se puede "esquivar" la matriz de $n \times n$ usando matrices de $n \times p$. También se puede usar, de forma equivalente,
$$
np.sum(A^T \odot B, axis=0).T
$$
queda a preferencia del alumno cuál usar.

Sean $A \in \mathbb{R}^{n \times p}$ y $B \in \mathbb{R}^{p \times n}$. El producto $A \cdot B$ da una matriz de $(n \times n)$.

El elemento $(i, j)$ de $A \cdot B$ es:

$$
(A \cdot B)_{ij} = \sum_{k=1}^{p} A_{ik} \cdot B_{kj}
$$

La diagonal es el caso particular $i = j$:

$$
(A \cdot B)_{ii} = \sum_{k=1}^{p} A_{ik} \cdot B_{ki}
$$

Por otro lado, $B^T \in \mathbb{R}^{n \times p}$ y su elemento $(i, k)$ es $(B^T)_{ik} = B_{ki}$.

El producto elemento a elemento $A \odot B^T$ tiene como elemento $(i, k)$:

$$
(A \odot B^T)_{ik} = A_{ik} \cdot (B^T)_{ik} = A_{ik} \cdot B_{ki}
$$

Sumando sobre las columnas (axis=1):

$$
\sum_{k=1}^{p} (A \odot B^T)_{ik} = \sum_{k=1}^{p} A_{ik} \cdot B_{ki} = (A \cdot B)_{ii}
$$

Que es exactamente el elemento $i$-ésimo de la diagonal. Por lo tanto:

$$
\text{diag}(A \cdot B) = \text{np.sum}(A \odot B^T, \text{axis}=1) \quad
$$

La ventaja es que $A \odot B^T$ tiene shape $(n, p)$ — nunca se construye la matriz $(n, n)$. Se pasa de $O(n^2 p)$ operaciones y $O(n^2)$ memoria a $O(np)$ en ambos casos.

Demostración de la forma equivalente: $\text{np.sum}(A^T \odot B, \text{axis}=0)^T$

Ahora $A^T \in \mathbb{R}^{p \times n}$ y $B \in \mathbb{R}^{p \times n}$, así que $A^T \odot B$ tiene shape $(p, n)$.

El elemento $(k, i)$ de $A^T \odot B$ es:

$$
(A^T \odot B)_{ki} = (A^T)_{ki} \cdot B_{ki} = A_{ik} \cdot B_{ki}
$$

Sumando sobre las filas (axis=0):

$$
\sum_{k=1}^{p} (A^T \odot B)_{ki} = \sum_{k=1}^{p} A_{ik} \cdot B_{ki} = (A \cdot B)_{ii}
$$

Esa suma da un vector fila de shape $(n,)$. El $.T$ al final es para mantener consistencia de shape (vector columna). El resultado es el mismo:

$$
\text{diag}(A \cdot B) = \text{np.sum}(A^T \odot B, \text{axis}=0)^T \quad
$$

Ambas formas evitan construir la matriz $(n, n)$ y tienen costo $O(np)$. La diferencia es solo si se trabaja con las matrices en su forma original $(p, n)$ o transpuesta $(n, p)$.

6. Utilizar la propiedad antes demostrada para reimplementar la predicción del modelo `FasterQDA` de forma eficiente en un nuevo modelo `EfficientQDA`.

In [26]:
# EfficientQDA usa diag(A·B) = sum(A ⊙ B^T, axis=1)
# para evitar construir la matriz n×n.
#
# En FasterQDA el cuello de botella era:
#   quad = U^T @ SU  → (k, n, n)   ← matriz n×n por clase
#   dist = diagonal(quad)          ← se descarta casi todo
#
# Aplicando la propiedad con A = U^T (k,n,p) y B = SU (k,p,n):
#   diag(U^T @ SU) = sum(U^T ⊙ SU^T, axis=1)
# Equivalentemente, trabajando en (k,p,n):
#   diag = sum(U * SU, axis=1)  → (k, n)  directamente
#
# Nunca se construye la matriz (k, n, n). Memoria: O(np) en vez de O(n²).

class EfficientQDA(TensorizedQDA):
    def _predict_log_conditionals(self, X):
        # X: (p, n), tensor_means: (k, p, 1), tensor_inv_cov: (k, p, p)

        # Centrar: (p, n) - (k, p, 1) → (k, p, n)
        U = X - self.tensor_means

        # Producto Σ_j⁻¹ @ U: (k, p, p) @ (k, p, n) → (k, p, n)
        SU = self.tensor_inv_cov @ U

        # diag(Uᵀ @ SU) = sum(U * SU, axis=1) → (k, n)
        # U y SU tienen shape (k, p, n), el * es element-wise,
        # y sum(axis=1) colapsa la dimensión p → producto punto por observación

        dist = np.sum(U * SU, axis=1)  # (k, n)

        # Log-posteriori: (k, 1) - (k, n) → (k, n)
        # LA.det(...) devuelve (k,), lo pasamos a (k,1) para broadcasting correcto.
        log_det = 0.5 * np.log(LA.det(self.tensor_inv_cov)).reshape(-1, 1)
        return log_det - 0.5 * dist

    def predict(self, X):


        log_post = (self.log_a_priori.reshape(-1, 1) + self._predict_log_conditionals(X))
        

        # Clase ganadora por observación: (n,) → (1, n)
        return np.argmax(log_post, axis=0).reshape(1, -1)

7. Comparar la performance de las 4 variantes de QDA implementadas hasta ahora (no Cholesky) ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

In [27]:
# Benchmark de las 4 variantes con Letters dataset
X_letter, y_letter = get_letters_dataset()
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

b_letter = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

to_bench = [QDA, TensorizedQDA, FasterQDA, EfficientQDA]

for model in to_bench:
    b_letter.bench(model)

summ_letter = b_letter.summary(baseline='QDA')
summ_letter[[
    'train_median_ms', 'test_median_ms', 'mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

FasterQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

FasterQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,4.025877,619.630688,0.886117,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,4.174297,124.983577,0.885303,0.964444,4.957697,1.001819,0.632780
FasterQDA,6.175026,1405.225140,0.884827,0.651961,0.440948,0.999853,0.000030
EfficientQDA,3.817801,12.365846,0.884890,1.054501,50.108233,0.998188,0.002497


### Diferencias entre implementaciones de `QDA_Chol`

8. Si una matriz $A$ tiene fact. de Cholesky $A=LL^T$, expresar $A^{-1}$ en términos de $L$. ¿Cómo podría esto ser útil en la forma cuadrática de QDA?

Si $A = LL^T$, entonces invertimos ambos lados:

$$
A^{-1} = (LL^T)^{-1} = (L^{-1})^T L^{-1}
$$

Porque la inversa de un producto se invierte en orden: $(AB)^{-1} = B^{-1}A^{-1}$.

En QDA esto permite reemplazar inversion directa por operaciones triangulares, mas eficientes y numericamente estables.

9. Explicar las diferencias entre `QDA_Chol1`y `QDA` y cómo `QDA_Chol1` llega, paso a paso, hasta las predicciones.

QDA trabaja directamente con la inversa de la covarianza $\Sigma_j^{-1}$: calcula `np.cov`, la invierte con `LA.inv`, y en la predicción hace la multiplicación matricial completa $(x-\mu_j)^T \Sigma_j^{-1} (x-\mu_j)$.

QDA_Chol1 en cambio descompone la covarianza con Cholesky: $\Sigma_j = L_j L_j^T$, e invierte $L_j$ con `LA.inv`. En la predicción hace algo distinto: calcula $y = L_j^{-1}(x - \mu_j)$ y después simplemente hace `(y**2).sum()`.

Esto funciona porque:

$$
y^T y = (L_j^{-1}(x-\mu_j))^T (L_j^{-1}(x-\mu_j)) = (x-\mu_j)^T \underbrace{(L_j^{-1})^T L_j^{-1}}_{= \Sigma_j^{-1}} (x-\mu_j)
$$

O sea, la norma: $\|y\|^2 = y^T y $ es exactamente la distancia, pero en vez de hacer una multiplicación matricial, se reduce a sumar los cuadrados de un vector. Es más barato.

Para el determinante también hay una simplificación. Como $L_j^{-1}$ es triangular, su determinante es el producto de la diagonal:

$$
\frac{1}{2}\log|\Sigma_j^{-1}| = \log\prod_i (L_j^{-1})_{ii}
$$

En el código esto es `np.log(L_inv.diagonal().prod())` — no necesita calcular `LA.det`.

10. ¿Cuáles son las diferencias entre `QDA_Chol1`, `QDA_Chol2` y `QDA_Chol3`?

- **QDA_Chol1**: calcula $L_j^{-1}$ explícitamente con `LA.inv` (inversión genérica) y guarda $L_j^{-1}$.
- **QDA_Chol2**: no invierte $L_j$. Guarda $L_j$ directamente y en la predicción resuelve el sistema $L_j y = (x - \mu_j)$ con `solve_triangular` (forward substitution)
- **QDA_Chol3**: como Chol1, calcula y guarda $L_j^{-1}$, pero usa `dtrtri` en vez de `LA.inv`. `dtrtri` es una rutina de LAPACK especializada para invertir matrices triangulares

11. Comparar la performance de las 7 variantes de QDA implementadas hasta ahora ¿Qué se observa?¿Hay alguna de las implementaciones de `QDA_Chol` que sea claramente mejor que las demás?¿Alguna que sea peor?

In [28]:
# Benchmark de las 7 variantes con Letters dataset
X_letter, y_letter = get_letters_dataset()
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

b_letter_7 = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

to_bench = [QDA, TensorizedQDA, FasterQDA, EfficientQDA, QDA_Chol1, QDA_Chol2, QDA_Chol3]

for model in to_bench:
    b_letter_7.bench(model)

summ_letter_7 = b_letter_7.summary(baseline='QDA')
summ_letter_7[[
    'train_median_ms', 'test_median_ms', 'mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

...

QDA_Chol2 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol2 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

QDA_Chol3 (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA_Chol3 (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,4.273718,614.632522,0.886117,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,4.276570,125.859828,0.885303,0.999333,4.883469,1.001819,0.631035
FasterQDA,6.059489,1362.366648,0.884827,0.705293,0.451151,1.000000,0.000030
EfficientQDA,3.816244,11.255364,0.884890,1.119875,54.607967,0.998188,0.002490
QDA_Chol1,4.532703,334.064737,0.884770,0.942863,1.839861,1.002047,1.026525
QDA_Chol2,3.856202,741.698992,0.885433,1.108271,0.828682,1.000994,1.028098
QDA_Chol3,4.028957,327.881225,0.885807,1.060750,1.874558,1.002588,1.028098


### Optimización

12. Implementar el modelo `TensorizedChol` paralelizando sobre clases/observaciones según corresponda. Se recomienda heredarlo de alguna de las implementaciones de `QDA_Chol`, aunque la elección de cuál de ellas queda a cargo del alumno según lo observado en los benchmarks de puntos anteriores.

In [29]:
# TensorizedChol hereda de QDA_Chol3 porque:
# - Chol1 y Chol3 tuvieron performance similar y ambas superaron a Chol2
# - Chol3 usa dtrtri (LAPACK, especializada para triangulares) que es
#   teóricamente mejor que LA.inv (genérica) de Chol1
#
# Como TensorizedQDA, solo tensoriza sobre las k clases.
# El for-loop sobre observaciones se mantiene (heredado de BaseBayesianClassifier).

class TensorizedChol(QDA_Chol3):

    def _fit_params(self, X, y):
        super()._fit_params(X, y)

        # Stack en tensores para operar sobre todas las clases a la vez
        self.tensor_L_inv = np.stack(self.L_invs)    # (k, p, p)
        self.tensor_means = np.stack(self.means)      # (k, p, 1)

        # Precomputar log-det: constante por clase, evita recalcularlo n×k veces
        self.log_dets = np.array([
            np.log(L_inv.diagonal().prod()) for L_inv in self.L_invs
        ])  # (k,)

    def _predict_log_conditionals(self, x):
        # Calcula log f_j(x) para las k clases a la vez (una sola observación)
        unbiased_x = x - self.tensor_means            # (k, p, 1)
        Y = self.tensor_L_inv @ unbiased_x            # (k, p, 1)
        return self.log_dets - 0.5 * (Y ** 2).sum(axis=1).flatten()  # (k,)

    def _predict_one(self, x):
        return np.argmax(self.log_a_priori + self._predict_log_conditionals(x))

13. Implementar el modelo `EfficientChol` combinando los insights de `EfficientQDA` y `TensorizedChol`. Si se desea, se puede implementar `FasterChol` como ayuda, pero no se contempla para el punto.

In [30]:
# EfficientChol combina:
# - De EfficientQDA: tensorizar sobre clases Y observaciones, evitando la matriz n×n
# - De Cholesky: la descomposición que convierte la forma cuadrática en una norma
#
# A diferencia de TensorizedChol (que solo tensoriza sobre clases y mantiene
# el for-loop sobre observaciones), EfficientChol elimina ambos for-loops.
#
# La conexión clave: en EfficientQDA usábamos
#   diag(Uᵀ Σ⁻¹ U) = sum(U * (Σ⁻¹ U), axis=1)
#
# Con Cholesky, Σ⁻¹ = (L⁻¹)ᵀ L⁻¹, entonces:
#   diag(Uᵀ (L⁻¹)ᵀ L⁻¹ U) = diag(Yᵀ Y)   donde Y = L⁻¹U
#
# Y diag(Yᵀ Y) se simplifica a:
#   sum(Y², axis=1)

class EfficientChol(TensorizedChol):
    # Hereda de TensorizedChol (que ya tiene _fit_params con tensores y log_dets)

    def predict(self, X):
        # X: (p, n) — todas las observaciones de golpe

        # Centrar: (p, n) - (k, p, 1) → (k, p, n)
        U = X - self.tensor_means

        # Y = L⁻¹ U: (k, p, p) @ (k, p, n) → (k, p, n)
        Y = self.tensor_L_inv @ U

        # diag(Yᵀ Y) = sum(Y², axis=1) → (k, n)
        dist = (Y ** 2).sum(axis=1)  # (k, n)

        # Log-posteriori: (k, 1) - (k, n) → (k, n)
        log_post = (self.log_a_priori + self.log_dets).reshape(-1, 1) - 0.5 * dist

        # Clase ganadora: (n,) → (1, n)
        return np.argmax(log_post, axis=0).reshape(1, -1)

13. Comparar la performance de las 9 variantes de QDA implementadas ¿Qué se observa? A modo de opinión ¿Se condice con lo esperado?

In [31]:
# Benchmark de las 9 variantes con Letters dataset
X_letter, y_letter = get_letters_dataset()
y_letter_encoded = label_encode(y_letter.reshape(-1,1))

b_letter_9 = Benchmark(
    X_letter, y_letter_encoded,
    same_splits=False,
    n_runs=100,
    warmup=20,
    mem_runs=30,
    test_sz=0.2
)

to_bench = [
    QDA, TensorizedQDA, FasterQDA, EfficientQDA,
    QDA_Chol1, QDA_Chol2, QDA_Chol3,
    TensorizedChol, EfficientChol
]

for model in to_bench:
    b_letter_9.bench(model)

summ_letter_9 = b_letter_9.summary(baseline='QDA')
summ_letter_9[[
    'train_median_ms', 'test_median_ms', 'mean_accuracy',
    'train_speedup', 'test_speedup',
    'train_mem_reduction', 'test_mem_reduction'
]]

Benching params:
Total runs: 150
Warmup runs: 20
Peak Memory usage runs: 30
Running time runs: 100
Train size rows (approx): 16000
Test size rows (approx): 4000
Test size fraction: 0.2


QDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

QDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

TensorizedQDA (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedQDA (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

...

TensorizedChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

TensorizedChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

EfficientChol (MEM):   0%|          | 0/30 [00:00<?, ?it/s]

EfficientChol (TIME):   0%|          | 0/100 [00:00<?, ?it/s]

,train_median_ms,test_median_ms,mean_accuracy,train_speedup,test_speedup,train_mem_reduction,test_mem_reduction
model,,,,,,,
QDA,4.163110,613.679358,0.886117,1.000000,1.000000,1.000000,1.000000
TensorizedQDA,3.982923,124.691294,0.885303,1.045240,4.921589,1.001819,0.631326
FasterQDA,5.590224,1356.999820,0.884827,0.744712,0.452232,1.000000,0.000030
EfficientQDA,3.480325,11.861533,0.884890,1.196184,51.736935,0.998188,0.002492
QDA_Chol1,4.575560,334.751080,0.884770,0.909858,1.833241,1.002047,1.026998
QDA_Chol2,4.335356,741.742335,0.885433,0.960269,0.827348,1.000994,1.028571
QDA_Chol3,4.067201,327.968167,0.885807,1.023581,1.871155,1.002588,1.028571
TensorizedChol,3.658274,23.959840,0.884995,1.137998,25.612832,1.001676,0.604919
EfficientChol,3.442962,8.884144,0.885720,1.209165,69.075800,1.001449,0.002492
